In [5]:
import pandas as pd
from sqlalchemy import create_engine

# --- 1. CONFIGURACIÓN DE LA BASE DE DATOS ---
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "192.168.100.50"
DB_PORT = "5433"
DB_NAME = "ingresos_db"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# --- 2. EXTRACCIÓN DE DATOS ---
# Nota: El doble signo de porcentaje (%%) se usa a veces en Python para escapar el '%' 
# en consultas SQL dentro de ciertos drivers, pero pandas suele aceptar el '%' simple.
query = """
    SELECT * FROM salidas 
    WHERE codigo LIKE '6-06-%%' 
    AND gerencia = 'GERENCIA DE OPERACION Y MANTENIMIENTO'
"""

print("Extrayendo datos de la base de datos...")

# Leemos directamente a un DataFrame
df = pd.read_sql(query, con=engine)


# Nos aseguramos de que la columna cantidad sea numérica para poder sumarla
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').fillna(0)

# --- 3. ANÁLISIS DE ROTACIÓN ---
print("\nProcesando análisis de rotación (frecuencia y volumen)...")

# Agrupamos por código e ítem para contar las salidas y sumar las cantidades
analisis_rotacion = df.groupby(['codigo', 'item']).agg(
    frecuencia_salidas=('codigo', 'count'),      # Veces que se solicitó
    cantidad_total=('cantidad', 'sum')           # Total de unidades consumidas
).reset_index()

# Ordenamos de mayor a menor frecuencia para ver los más solicitados primero
analisis_rotacion = analisis_rotacion.sort_values(by='frecuencia_salidas', ascending=False)

# --- 4. RESULTADOS ---
print("\n" + "="*70)
print(" TOP INSUMOS QUÍMICOS DE MAYOR ROTACIÓN (Por Frecuencia de Salida)")
print("="*70)

# Mostramos los 15 insumos con mayor movimiento
top_15 = analisis_rotacion.head(15)

# Formateamos la salida en la consola
for index, row in top_15.iterrows():
    print(f"Código: {row['codigo']}")
    print(f"Ítem:   {row['item']}")
    print(f"        -> Frecuencia de retiros: {row['frecuencia_salidas']} veces")
    print(f"        -> Volumen total retirado: {row['cantidad_total']} unidades")
    print("-" * 50)
    
# Opcional: Exportar el resultado a un archivo Excel para revisión formal
analisis_rotacion.to_excel("rotacion_quimicos_mantenimiento.xlsx", index=False)
print("\nReporte exportado exitosamente a 'rotacion_quimicos_mantenimiento.xlsx'")


Extrayendo datos de la base de datos...

Procesando análisis de rotación (frecuencia y volumen)...

 TOP INSUMOS QUÍMICOS DE MAYOR ROTACIÓN (Por Frecuencia de Salida)
Código: 6-06-00050
Ítem:   MARCADOR DE PINTURA PARA METAL (COLOR ROJO)
        -> Frecuencia de retiros: 81 veces
        -> Volumen total retirado: 1152.0 unidades
--------------------------------------------------
Código: 6-06-00028
Ítem:   PINTURA EN AEROSOL COLOR PLOMO ENVASE DE 8 OZ
        -> Frecuencia de retiros: 79 veces
        -> Volumen total retirado: 1265.0 unidades
--------------------------------------------------
Código: 6-06-00063
Ítem:   LIMPIADOR MULTIUSO AFLOJA TODO 400 ML
        -> Frecuencia de retiros: 79 veces
        -> Volumen total retirado: 1016.0 unidades
--------------------------------------------------
Código: 6-06-00026
Ítem:   PINTURA EN AEROSOL COLOR ROJO CLARO ENVASE DE 8 OZ
        -> Frecuencia de retiros: 60 veces
        -> Volumen total retirado: 294.0 unidades
------------------

In [14]:
import pandas as pd
from sqlalchemy import create_engine

# --- 1. CONFIGURACIÓN DE LA BASE DE DATOS ---
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "192.168.100.50"
DB_PORT = "5433"
DB_NAME = "ingresos_db"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

def analizar_tiempo_entre_salidas():
    print("Extrayendo historial de salidas...")
    
    # Traemos solo las columnas necesarias para hacer el código más rápido
    query = """
        SELECT codigo, item, fecha_salida 
        FROM salidas 
        WHERE codigo IS NOT NULL AND fecha_salida IS NOT NULL
        AND codigo LIKE '6-06-%%' 
        AND gerencia = 'GERENCIA DE OPERACION Y MANTENIMIENTO'
    """
    
    try:
        df = pd.read_sql(query, con=engine)
        
        if df.empty:
            print("No hay datos suficientes para analizar.")
            return

        # 1. Convertir la columna a formato de fecha (Datetime)
        # Asumimos formato día/mes/año según tus archivos anteriores
        df['fecha_salida'] = pd.to_datetime(df['fecha_salida'], format='%d/%m/%Y', errors='coerce')
        
        # Eliminar filas donde la fecha no se pudo convertir
        df = df.dropna(subset=['fecha_salida'])

        # 2. Ordenar por Código de Ítem y luego por Fecha (de más antigua a más nueva)
        df = df.sort_values(by=['codigo', 'fecha_salida'])

        # 3. Calcular los días transcurridos entre una salida y la anterior DEL MISMO ÍTEM
        # .diff() resta la fecha actual con la fila anterior
        df['dias_transcurridos'] = df.groupby('codigo')['fecha_salida'].diff().dt.days

        print("\nProcesando promedios de frecuencia...")
        
        # 4. Agrupar para calcular el promedio de días y el total de veces que salió
        analisis = df.groupby(['codigo', 'item']).agg(
            total_veces_solicitado=('fecha_salida', 'count'),
            promedio_dias_entre_salidas=('dias_transcurridos', 'mean') # Promedio de los intervalos
        ).reset_index()

        # 5. Filtrar y limpiar
        # Solo podemos tener promedio si salió 2 o más veces. 
        analisis = analisis[analisis['total_veces_solicitado'] >= 2]
        
        # Redondear el promedio de días a 1 decimal para que sea legible
        analisis['promedio_dias_entre_salidas'] = analisis['promedio_dias_entre_salidas'].round(1)

        # 6. Ordenar por los repuestos que salen MÁS RÁPIDO (menor cantidad de días)
        analisis = analisis.sort_values(by='promedio_dias_entre_salidas', ascending=True)

        # --- MOSTRAR RESULTADOS ---
        print("\n" + "="*80)
        print(" REPUESTOS CON MAYOR VELOCIDAD DE SALIDA (FRECUENCIA EN DÍAS)")
        print("="*80)
        
        #top_20 = analisis.head(20)
        
        display(analisis)
        analisis.to_excel("frecuencia_salidas.xlsx", index=False)
        print("\nArchivo Excel generado: frecuencia_salidas.xlsx")
            
        # analisis.to_excel("frecuencia_salidas.xlsx", index=False)
        # print("\nArchivo Excel generado: frecuencia_salidas.xlsx")

    except Exception as e:
        print(f"Ocurrió un error crítico: {e}")

# Ejecutar la función
analizar_tiempo_entre_salidas()

Extrayendo historial de salidas...

Procesando promedios de frecuencia...

 REPUESTOS CON MAYOR VELOCIDAD DE SALIDA (FRECUENCIA EN DÍAS)


,codigo,item,total_veces_solicitado,promedio_dias_entre_salidas
7,6-06-00008,"PINTURA SINTETICA MATE, COLOR AMARILLO DE 3.5LTS.",2,0.0
60,6-06-00069,PINTURA EN SPRAY,3,0.0
50,6-06-00059,PINTURA EN AEROSOL COLOR AZUL TUBO 400 ML,3,0.0
95,6-06-00122,LIQUIDO ANTIGEL 35% N65%H20 30L,2,0.0
109,6-06-00163,SILICONA DE ALTA TEMPERATURA MEGA,2,0.0
...,...,...,...,...
17,6-06-00018,JERINGA DE 10ML,6,526.4
85,6-06-00110,PINTURA A BASE DE NITROCELULOSA Y RESINA SINTE...,2,567.0
62,6-06-00071,LIMPIADOR DE FRENOS,3,628.0
23,6-06-00029,ALCOHOL ISOPROPILICO DE 1 LITRO,2,730.0



Archivo Excel generado: frecuencia_salidas.xlsx


In [16]:
import pandas as pd
from sqlalchemy import create_engine

# --- 1. CONFIGURACIÓN DE LA BASE DE DATOS ---

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

def analizar_frecuencia_por_linea():
    print("Extrayendo historial de salidas por línea...")
    
    # Asumimos que tienes una columna llamada 'linea' en tu base de datos.
    # Si el dato está dentro de 'ubicacion' o 'sistema', asegúrate de cambiar el nombre aquí.
    query = """
        SELECT linea, codigo, item, fecha_salida 
        FROM salidas 
        WHERE codigo IS NOT NULL 
          AND fecha_salida IS NOT NULL
          AND linea IS NOT NULL
          AND codigo LIKE '6-06-%%' 
        AND gerencia = 'GERENCIA DE OPERACION Y MANTENIMIENTO'
    """
    
    try:
        df = pd.read_sql(query, con=engine)
        
        if df.empty:
            print("No hay datos suficientes para analizar con la columna 'linea'.")
            return

        # 1. Convertir fecha (Asegurar formato correcto)
        df['fecha_salida'] = pd.to_datetime(df['fecha_salida'], format='%d/%m/%Y', errors='coerce')
        df = df.dropna(subset=['fecha_salida'])

        # 2. Ordenar por Línea -> Código de Ítem -> Fecha
        df = df.sort_values(by=['linea', 'codigo', 'fecha_salida'])

        # 3. Calcular días transcurridos agrupando por LÍNEA y CÓDIGO
        # Esto evita que mezclemos fechas de la Línea Roja con las de la Línea Amarilla para el mismo ítem
        df['dias_transcurridos'] = df.groupby(['linea', 'codigo'])['fecha_salida'].diff().dt.days

        print("\nProcesando promedios de frecuencia por línea...")
        
        # 4. Agrupar para calcular métricas
        analisis = df.groupby(['linea', 'codigo', 'item']).agg(
            total_salidas=('fecha_salida', 'count'),
            promedio_dias=('dias_transcurridos', 'mean')
        ).reset_index()

        # 5. Filtrar ítems que salieron al menos 2 veces (requerido para un intervalo)
        analisis = analisis[analisis['total_salidas'] >= 2]
        
        # Redondear días
        analisis['promedio_dias'] = analisis['promedio_dias'].round(1)

        # 6. Ordenar por Línea (alfabético) y luego por los que salen más rápido
        analisis = analisis.sort_values(by=['linea', 'promedio_dias'], ascending=[True, True])

        # --- MOSTRAR RESULTADOS ---
        print("\n" + "="*90)
        print(" ANÁLISIS DE ROTACIÓN: FRECUENCIA DE SALIDAS SEPARADO POR LÍNEA")
        print("="*90)
        
        # Mostrar agrupado por línea para mejor lectura en consola
        lineas_actuales = analisis['linea'].unique()
        
        for linea in lineas_actuales:
            print(f"\n📍 {linea.upper()}")
            print("-" * 60)
            
            # Filtramos los top 10 ítems más rápidos de esta línea específica
            top_linea = analisis[analisis['linea'] == linea].head(10)
            
            for index, row in top_linea.iterrows():
                print(f"  [{row['codigo']}] {row['item']}")
                print(f"     -> Promedio: Cada {row['promedio_dias']} días (Total: {row['total_salidas']} salidas)")
            
        # Opcional: Exportar a Excel
        analisis.to_excel("frecuencia_por_linea.xlsx", index=False)
        print("\n✅ Reporte exportado a 'frecuencia_por_linea.xlsx'")

    except Exception as e:
        print(f"Ocurrió un error crítico: {e}")

# Ejecutar
analizar_frecuencia_por_linea()

Extrayendo historial de salidas por línea...

Procesando promedios de frecuencia por línea...

 ANÁLISIS DE ROTACIÓN: FRECUENCIA DE SALIDAS SEPARADO POR LÍNEA

📍 AMARILLA
------------------------------------------------------------
  [6-06-00034] THINNER UNIVERSAL 3030 DE 5 LITROS
     -> Promedio: Cada 0.0 días (Total: 2 salidas)
  [6-06-00069] PINTURA EN SPRAY 
     -> Promedio: Cada 0.0 días (Total: 2 salidas)
  [6-06-00053] MARCADOR DE PINTURA PARA METAL COLOR AZUL
     -> Promedio: Cada 4.0 días (Total: 2 salidas)
  [6-06-00050] MARCADOR DE PINTURA PARA METAL (COLOR ROJO)
     -> Promedio: Cada 135.3 días (Total: 4 salidas)
  [6-06-00051] MARCADOR DE PINTURA PARA METAL (COLOR AMARILLO)
     -> Promedio: Cada 203.0 días (Total: 2 salidas)
  [6-06-00028] PINTURA EN AEROSOL COLOR PLOMO ENVASE DE 8 OZ
     -> Promedio: Cada 310.7 días (Total: 4 salidas)
  [6-06-00001] SILICONA WHITE PARA TABLERO 300ML. VISTONY
     -> Promedio: Cada 359.0 días (Total: 2 salidas)
  [6-06-00003] PINTURA

In [3]:
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine

# --- 1. CONFIGURACIÓN DE LA BASE DE DATOS ---
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "192.168.100.50"
DB_PORT = "5433"
DB_NAME = "ingresos_db"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

def proyeccion_demanda_anual():
    print("Extrayendo historial de consumos (salidas)...")
    
    query = """
        SELECT codigo, item, cantidad, fecha_salida 
        FROM salidas 
        WHERE codigo IS NOT NULL 
          AND fecha_salida IS NOT NULL
          AND codigo LIKE '6-06-%%' 
        AND gerencia = 'GERENCIA DE OPERACION Y MANTENIMIENTO'
    """
    
    try:
        # Cargar datos
        df = pd.read_sql(query, con=engine)
        
        if df.empty:
            print("No hay datos suficientes para analizar.")
            return

        # 1. Limpieza y conversión de tipos
        df['fecha_salida'] = pd.to_datetime(df['fecha_salida'], format='%d/%m/%Y', errors='coerce')
        df = df.dropna(subset=['fecha_salida'])
        df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').fillna(0)

        # 2. Determinar la "Ventana de Observación" (Días del periodo analizado)
        fecha_min = df['fecha_salida'].min()
        fecha_max = df['fecha_salida'].max()
        dias_periodo = (fecha_max - fecha_min).days + 1  # +1 para incluir el día de inicio
        
        print(f"Periodo analizado: {dias_periodo} días (Del {fecha_min.strftime('%d/%m/%Y')} al {fecha_max.strftime('%d/%m/%Y')})")

        # 3. Agrupar por ítem para obtener el consumo total en ese periodo
        consumo = df.groupby(['codigo', 'item']).agg(
            consumo_periodo=('cantidad', 'sum')
        ).reset_index()

        # 4. CÁLCULO DE LA NECESIDAD ANUAL PROYECTADA
        # Fórmula: (Consumo en el periodo / Días del periodo) * 365 días
        consumo['necesidad_anual'] = (consumo['consumo_periodo'] / dias_periodo) * 365
        
        # Redondear a números enteros (no puedes pedir medio repuesto)
        consumo['necesidad_anual'] = consumo['necesidad_anual'].round(0).astype(int)

        # 5. Ordenar para obtener los ítems que requerirán mayor volumen de compra
        consumo = consumo.sort_values(by='necesidad_anual', ascending=False)
        
        consumo.to_excel("proyeccion_demanda_anual_2.xlsx", index=False)
        print("\nArchivo Excel generado: proyeccion_demanda_anual.xlsx")
        
        top_20 = consumo.head(20)

        print("\nCalculando proyección y generando gráfico...")

        # --- 6. GENERAR GRÁFICO CON PLOTLY ---
        # Recortar nombres muy largos para que se vean bien en el gráfico
        top_20['item_corto'] = top_20['item'].apply(lambda x: (x[:45] + '...') if len(x) > 45 else x)

        fig = px.bar(
            top_20, 
            x='item_corto', 
            y='necesidad_anual',
            text='necesidad_anual', # Muestra el número sobre la barra
            color='necesidad_anual', # Da un degradado de color según el volumen
            color_continuous_scale='Blues',
            title=f"PROYECCIÓN DE NECESIDAD DE PEDIDO ANUAL (Top 20 Ítems)<br><sup>Basado en un historial de consumo de {dias_periodo} días</sup>",
            labels={'item_corto': 'Repuesto / Insumo', 'necesidad_anual': 'Unidades Estimadas por Año'}
        )

        fig.update_traces(textposition='outside')
        fig.update_layout(
            font=dict(family="Arial Narrow"),
            xaxis_tickangle=-45, # Inclina los nombres para mejor lectura
            margin=dict(b=150),  # Margen inferior para nombres largos
            plot_bgcolor='rgba(240,240,240,0.5)'
        )

        # Abrir el gráfico en el navegador web
        fig.show()

        print("\n¡Gráfico generado exitosamente en tu navegador!")

    except Exception as e:
        print(f"Ocurrió un error crítico: {e}")

# Ejecutar la función
proyeccion_demanda_anual()

Extrayendo historial de consumos (salidas)...
Periodo analizado: 3269 días (Del 26/04/2017 al 07/04/2026)

Archivo Excel generado: proyeccion_demanda_anual.xlsx

Calculando proyección y generando gráfico...


C:\Users\mantenimiento\AppData\Local\Temp\ipykernel_17072\2514300953.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20['item_corto'] = top_20['item'].apply(lambda x: (x[:45] + '...') if len(x) > 45 else x)



¡Gráfico generado exitosamente en tu navegador!
